In [50]:
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.13.0"

using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using Plotly.NET;
using Plotly.NET.LayoutObjects;
using Plotly.NET.CSharp;
using Chart = Plotly.NET.CSharp.Chart;

double[] xTrain, yTrain, xTest, yTest;

if (File.Exists("lab_1_train.csv") && File.Exists("lab_1_test.csv"))
{
    var trainLines = File.ReadAllLines("lab_1_train.csv").Skip(1).Where(l => !string.IsNullOrWhiteSpace(l));
    xTrain = trainLines.Select(l => double.Parse(l.Split(',')[1], CultureInfo.InvariantCulture)).ToArray();
    yTrain = trainLines.Select(l => double.Parse(l.Split(',')[2], CultureInfo.InvariantCulture)).ToArray();

    var testLines = File.ReadAllLines("lab_1_test.csv").Skip(1).Where(l => !string.IsNullOrWhiteSpace(l));
    xTest = testLines.Select(l => double.Parse(l.Split(',')[1], CultureInfo.InvariantCulture)).ToArray();
    yTest = testLines.Select(l => double.Parse(l.Split(',')[2], CultureInfo.InvariantCulture)).ToArray();
}
else
{
    Console.WriteLine("ERROR :<");
    return;
}

double wTrue = 2.0;
double bTrue = 20.0;

double w = 0.0;
double b = 0.0;
double lr = 0.01;
double lambda = 0.0;
int countEpoch = 10000; 

int nTrain = xTrain.Length; 
int nTest = xTest.Length;

int logStep = Math.Max(1, countEpoch / 1000);

var tableEpochs = new HashSet<int>
{
    1,
    Math.Max(1, (int)(countEpoch * 0.005)),
    Math.Max(1, (int)(countEpoch * 0.025)), 
    Math.Max(1, (int)(countEpoch * 0.10)),  
    Math.Max(1, (int)(countEpoch * 0.25)),  
    Math.Max(1, (int)(countEpoch * 0.50)), 
    countEpoch                              
};

var epochsHistory = new List<double>();
var trainLossHistory = new List<double>();
var testLossHistory = new List<double>();
var tableRows = new List<(int Epoch, double W, double B, double CostTrain, double CostTest, double BiasSq, double Variance)>();

for (int epoch = 1; epoch <= countEpoch; epoch++)
{
    double sumErrorSq = 0.0;
    double dw = 0.0;
    double db = 0.0;

    for (int i = 0; i < nTrain; i++)
    {
        double pred = w * xTrain[i] + b;
        double error = pred - yTrain[i];

        sumErrorSq += error * error;
        dw += error * xTrain[i];
        db += error;
    }

    double costTrain = sumErrorSq / nTrain + (lambda / nTrain) * (w * w);

    dw = (2.0 * dw / nTrain) + (2.0 * lambda / nTrain) * w;
    db = 2.0 * db / nTrain;

    w -= lr * dw;
    b -= lr * db;

    double testSumErrorSq = 0.0;
    double biasSq = 0.0;

    for (int i = 0; i < nTest; i++)
    {
        double pred = w * xTest[i] + b;
        double yStar = wTrue * xTest[i] + bTrue;
        double error = pred - yTest[i];

        testSumErrorSq += error * error;
        biasSq += Math.Pow(pred - yStar, 2);
    }

    double costTest = testSumErrorSq / nTest + (lambda / nTest) * (w * w);    biasSq /= nTest;
    double mseTest = testSumErrorSq / nTest;
    double variance = Math.Max(0, mseTest - biasSq);

    if (epoch % logStep == 0 || epoch == 1 || epoch == countEpoch)
    {
        epochsHistory.Add(epoch);
        trainLossHistory.Add(costTrain);
        testLossHistory.Add(costTest);
    }

    if (tableEpochs.Contains(epoch))
    {
        tableRows.Add((epoch, w, b, costTrain, costTest, biasSq, variance));
    }
}

Console.WriteLine(new string('-', 92));
Console.WriteLine($"| {"Епоха",-8} | {"w",-8} | {"b",-8} | {"Cost Train",-11} | {"Cost Test",-11} | {"Bias²",-11} | {"Variance",-11} |");
Console.WriteLine(new string('-', 92));

foreach (var row in tableRows.OrderBy(r => r.Epoch))
{
    Console.WriteLine($"| {row.Epoch,-8} | {row.W,-8:F3} | {row.B,-8:F3} | {row.CostTrain,-11:F4} | {row.CostTest,-11:F4} | {row.BiasSq,-11:F4} | {row.Variance,-11:F4} |");
}
Console.WriteLine(new string('-', 92));

var trainPoints = Chart.Point<double, double, string>(x: xTrain, y: yTrain, Name: "Train Data");
var testPoints = Chart.Point<double, double, string>(x: xTest, y: yTest, Name: "Test Data");
double minX = Math.Min(xTrain.Min(), xTest.Min());
double maxX = Math.Max(xTrain.Max(), xTest.Max());
var regLine = Chart.Line<double, double, string>(x: new[] { minX, maxX }, y: new[] { w * minX + b, w * maxX + b }, Name: "h(x) = w*x + b");

var chart1Fit = Chart.Combine(new[] { trainPoints, testPoints, regLine })
    .WithXAxisStyle<double, double, string>(Title: Title.init("x"))
    .WithYAxisStyle<double, double, string>(Title: Title.init("y"))
    .WithTitle($"Univariate Linear Regression | w: {w:F3}, b: {b:F3}");

int zoomStartEpoch = Math.Max(2, (int)(countEpoch * 0.01));
var zoomFilter = epochsHistory.Select((ep, idx) => new { ep, idx }).Where(x => x.ep >= zoomStartEpoch).ToList();
var epochsZoom = zoomFilter.Select(x => x.ep).ToArray();

var chartTrainLoss = Chart.Line<double, double, string>(x: epochsZoom, y: zoomFilter.Select(x => trainLossHistory[x.idx]).ToArray(), Name: "Train Cost J(w,b)");
var chartTestLoss = Chart.Line<double, double, string>(x: epochsZoom, y: zoomFilter.Select(x => testLossHistory[x.idx]).ToArray(), Name: "Test Cost J(w,b)");

var chart2Loss = Chart.Combine(new[] { chartTrainLoss, chartTestLoss })
    .WithXAxisStyle<double, double, string>(Title: Title.init($"Epoch (>= {zoomStartEpoch})"))
    .WithYAxisStyle<double, double, string>(Title: Title.init("Cost Function J(w,b)"))
    .WithTitle($"Gradient Descent: Cost Function J(w,b) Convergence");

chart1Fit.Display();
chart2Loss.Display();

Installed Packages Plotly.NET, 5.0.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

--------------------------------------------------------------------------------------------
| Епоха    | w        | b        | Cost Train  | Cost Test   | Bias²       | Variance    |
--------------------------------------------------------------------------------------------
| 1        | 0.124    | 0.411    | 422.4774    | 444.7955    | 445.0786    | 0.0000      |
| 50       | 3.747    | 12.595   | 49.0188     | 36.0434     | 36.0633     | 0.0000      |
| 250      | 5.245    | 18.876   | 0.4124      | 2.4486      | 2.3391      | 0.1095      |
| 1000     | 4.139    | 19.298   | 0.2163      | 1.1905      | 1.0927      | 0.0978      |
| 2500     | 2.910    | 19.674   | 0.0997      | 0.2587      | 0.1754      | 0.0833      |
| 5000     | 2.208    | 19.890   | 0.0746      | 0.0788      | 0.0038      | 0.0750      |
| 10000    | 1.993    | 19.956   | 0.0730      | 0.0750      | 0.0025      | 0.0725      |
--------------------------------------------------------------------------------------

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->